# Run Overview

`research/data/runs/{run_id}` に保存された sweep run を読み、ランキング・軸別マージナル・window安定性を確認する入口ノートです。重いバックテスト計算は行いません。

In [ ]:
# Parameters
RUN_ID = "latest"
RUNS_ROOT = "research/data/runs"
METRIC = "return_to_dd"
TOP = 20


In [ ]:
from collections import defaultdict
import json
from IPython.display import Markdown, display

from research.src.store.trial_store import TrialStore
from research.src.store.views import format_table, marginal_by_axis, rank

store = TrialStore(RUNS_ROOT)
run_id = store.resolve_run_id(RUN_ID)
manifest = store.load_manifest(run_id)
rows = store.load_trials(run_id)
display(Markdown(f"## Run: `{run_id}`"))
display(Markdown("```json\n" + json.dumps({
    "spec": manifest.get("spec", {}).get("name"),
    "git_sha": manifest.get("git_sha"),
    "dataset": manifest.get("dataset"),
    "trial_count": manifest.get("trial_count"),
    "error_count": manifest.get("error_count"),
    "keep_trades": manifest.get("keep_trades"),
    "trade_files_count": manifest.get("trade_files_count"),
    "started_at": manifest.get("started_at"),
    "finished_at": manifest.get("finished_at"),
}, ensure_ascii=False, indent=2) + "\n```"))


In [ ]:
ranked = rank(rows, by=METRIC, top_k=TOP)
table_rows = []
for index, row in enumerate(ranked, start=1):
    summary = row.get("summary", {})
    tags = row.get("tags", {})
    window = row.get("window", {})
    table_rows.append({
        "rank": index,
        "trial_id": row.get("trial_id"),
        "case_name": tags.get("case_name"),
        "window": window.get("window_id"),
        METRIC: summary.get(METRIC),
        "total_scaled_pnl_pct": summary.get("total_scaled_pnl_pct"),
        "closed_trades": summary.get("closed_trades"),
        "max_drawdown_pct_points": summary.get("max_drawdown_pct_points"),
        "win_rate_pct": summary.get("win_rate_pct"),
        "second_half_scaled_pnl_pct": summary.get("second_half_scaled_pnl_pct"),
    })
display(Markdown("## Ranking"))
display(Markdown("```\n" + format_table(
    table_rows,
    [
        "rank",
        "trial_id",
        "case_name",
        "window",
        METRIC,
        "total_scaled_pnl_pct",
        "closed_trades",
        "max_drawdown_pct_points",
        "win_rate_pct",
        "second_half_scaled_pnl_pct",
    ],
) + "\n```"))


In [ ]:
display(Markdown("## Axis marginal"))
marginal_rows = marginal_by_axis(rows, metric=METRIC)
display(Markdown("```\n" + format_table(
    marginal_rows,
    ["axis", "value", "count", f"mean_{METRIC}", f"min_{METRIC}", f"max_{METRIC}"],
) + "\n```"))


In [ ]:
display(Markdown("## Window stability"))
by_case = defaultdict(list)
for row in rows:
    if row.get("error"):
        continue
    tags = row.get("tags", {})
    summary = row.get("summary", {})
    by_case[tags.get("case_name", row.get("trial_id"))].append(summary.get("total_scaled_pnl_pct"))
stability_rows = []
for case_name, values in by_case.items():
    clean = [value for value in values if isinstance(value, (int, float))]
    if not clean:
        continue
    stability_rows.append({
        "case_name": case_name,
        "windows": len(clean),
        "min_total_scaled_pnl_pct": min(clean),
        "mean_total_scaled_pnl_pct": sum(clean) / len(clean),
        "max_total_scaled_pnl_pct": max(clean),
    })
stability_rows = sorted(
    stability_rows,
    key=lambda item: item["mean_total_scaled_pnl_pct"],
    reverse=True,
)[:TOP]
display(Markdown("```\n" + format_table(
    stability_rows,
    ["case_name", "windows", "min_total_scaled_pnl_pct", "mean_total_scaled_pnl_pct", "max_total_scaled_pnl_pct"],
) + "\n```"))


In [ ]:
display(Markdown("## Errors"))
error_rows = [
    {
        "trial_id": row.get("trial_id"),
        "case_name": row.get("tags", {}).get("case_name"),
        "error": row.get("error"),
    }
    for row in rows
    if row.get("error")
]
display(Markdown("```\n" + format_table(error_rows, ["trial_id", "case_name", "error"]) + "\n```"))
